<a href="https://www.kaggle.com/code/saibhossain/build-standard-naive-vanilla-rag-tutorial?scriptVersionId=310292822" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Install packages

In [1]:
!pip -q install langchain langchain-huggingface faiss-cpu sentence-transformers pypdf
!pip install langchain-community langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain_core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,

In [2]:
!pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.1 MB/s eta 0:00:00


# Imports

In [3]:
import os
from typing import List
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI

# Load API key

In [4]:
from kaggle_secrets import UserSecretsClient
from dotenv import load_dotenv
load_dotenv()
user_secrets = UserSecretsClient()
os.environ["GOOGLE_API_KEY"] = user_secrets.get_secret("GOOGLE_API_KEY")

# Create a sample dataset source

In [5]:
raw_docs = [
    """
    Retrieval-Augmented Generation (RAG) is a framework that combines information retrieval
    with large language model generation. It improves factual accuracy by grounding responses
    in external documents.
    """,
    """
    Naive RAG typically includes document chunking, embedding generation, vector storage,
    top-k retrieval, and answer generation using retrieved context.
    """,
    """
    A limitation of naive RAG is that it may retrieve irrelevant chunks if the embeddings
    do not capture the true intent of the query.
    """
]

documents = [Document(page_content=text, metadata={"source": f"doc_{i}"}) for i, text in enumerate(raw_docs)]
print(f"Loaded {len(documents)} documents")

Loaded 3 documents


### more document

In [6]:
raw_docs = [
    """
    Retrieval-Augmented Generation (RAG) is a framework that combines information retrieval
    with large language model generation. It improves factual accuracy by grounding responses
    in external documents.
    """,
    """
    Naive RAG typically includes document chunking, embedding generation, vector storage,
    top-k retrieval, and answer generation using retrieved context.
    """,
    """
    A limitation of naive RAG is that it may retrieve irrelevant chunks if the embeddings
    do not capture the true intent of the query.
    """,
    """
    In recent years, discussions around artificial intelligence have expanded beyond core
    technical architectures into broader interdisciplinary domains. While Retrieval-Augmented
    Generation (RAG) is often presented as a solution to hallucination problems in language
    models, its practical implementation reveals a number of subtle challenges that are not
    always obvious in simplified tutorials.

    One such challenge arises when documents contain overlapping semantic signals. For example,
    a paragraph discussing “retrieval efficiency” might also include unrelated mentions of
    database indexing strategies, distributed systems, or even anecdotal examples from
    unrelated industries like logistics or healthcare. When embeddings are generated, these
    mixed signals can cause the vector representation to drift away from the primary intent
    of the text. As a result, naive similarity search may rank such chunks higher than more
    directly relevant content.

    Consider a scenario where a system is asked about “limitations of naive RAG.” Ideally,
    the retriever should prioritize chunks that explicitly discuss weaknesses such as poor
    context selection, embedding mismatch, or lack of query understanding. However, if a
    document includes general discussions about machine learning pipelines, optimization
    techniques, or even unrelated topics like cloud deployment costs, the embedding space
    may cluster these together in unexpected ways.

    This issue becomes more pronounced when documents are long and contain multiple themes.
    For instance, a single chunk might start with an explanation of RAG, transition into a
    discussion about neural network training, and end with a comparison of programming
    languages such as Python, JavaScript, and C++. While all of these are broadly related
    to technology, they are not equally relevant to a specific query about retrieval quality.
    Yet, naive chunking strategies often fail to separate these themes effectively.

    Beyond technical content, noise can also come from stylistic elements. Blog posts, for
    example, frequently include introductions, personal opinions, or storytelling elements
    that are only loosely connected to the main topic. A paragraph might begin with a story
    about a developer debugging a production issue, segue into a general explanation of APIs,
    and only briefly mention RAG systems. Embeddings generated from such text may emphasize
    the narrative aspects rather than the technical insight.

    Another source of confusion is the presence of repeated keywords across different contexts.
    Words like “model,” “training,” “data,” and “performance” appear in many areas of machine
    learning. Without deeper semantic understanding, embedding models may treat these as strong
    signals of relevance even when the surrounding context differs significantly. This can lead
    to retrieval results that feel superficially related but fail to answer the user’s question.

    The problem is further compounded when the dataset includes content from multiple domains.
    Imagine combining documents about RAG with materials on web development, mobile apps,
    cybersecurity, and even unrelated fields like finance or biology. A chunk discussing
    “data pipelines” in a financial context might be retrieved for a query about RAG pipelines,
    simply because of shared terminology. Similarly, discussions about “retrieval” in the
    context of search engines or database queries might interfere with RAG-specific retrieval.
    """,
    """
    In educational datasets, this challenge becomes even more apparent. Students often write
    notes that mix definitions, examples, and unrelated commentary. A single note might include
    a definition of RAG, followed by a comparison with GANs, and then a completely unrelated
    section about exam preparation strategies. If such notes are ingested without careful
    preprocessing, the retriever may surface irrelevant sections during inference.

    Additionally, naive RAG systems typically rely on static chunk sizes. This can lead to
    situations where important context is split across chunks, reducing its usefulness, while
    less relevant information is grouped together and retrieved as a single unit. For example,
    a detailed explanation of RAG limitations might be split into two chunks, each of which
    appears incomplete on its own. Meanwhile, a chunk containing a mix of unrelated topics
    might be retrieved because it happens to contain a few matching keywords.

    There are also challenges related to query formulation. Users often ask vague or ambiguous
    questions, such as “Why does RAG fail?” Without additional context, the retriever must
    guess which aspects of failure are relevant—retrieval quality, generation accuracy,
    system latency, or something else entirely. In such cases, noisy or off-topic chunks may
    be selected simply because they partially match the query.

    Interestingly, even improvements like increasing the number of retrieved documents (top-k)
    can introduce new problems. While retrieving more chunks increases the chance of including
    relevant information, it also raises the likelihood of including irrelevant or misleading
    content. The language model must then filter and synthesize this information, which can
    lead to inconsistent or diluted answers.
    """,
    """
    Another dimension to consider is the embedding model itself. Different models capture
    semantics in different ways, and some may struggle with domain-specific language or
    nuanced distinctions. For example, a model trained primarily on general web text might
    not effectively differentiate between “retrieval” in RAG and “retrieval” in database
    systems. This can further blur the boundaries between relevant and irrelevant chunks.

    In practice, addressing these issues requires more advanced techniques such as better
    chunking strategies, query rewriting, reranking, and hybrid retrieval methods. However,
    in a naive RAG setup, these enhancements are typically absent, making the system more
    susceptible to confusion.

    To summarize, while RAG offers a powerful framework for improving the factual grounding
    of language models, naive implementations are vulnerable to a variety of issues stemming
    from noisy data, mixed contexts, and imperfect embeddings. These challenges highlight
    the importance of careful dataset design and retrieval optimization.

    Outside the scope of RAG, it is also worth noting that modern software systems often
    integrate multiple components such as APIs, microservices, and cloud infrastructure.
    Discussions about these topics may appear alongside RAG in technical documents, further
    increasing the complexity of the retrieval task. For example, a paragraph about deploying
    a machine learning model on AWS might include references to storage, networking, and
    scaling, none of which are directly relevant to RAG but may still influence the embedding.

    Finally, as AI systems continue to evolve, the boundaries between different techniques
    are becoming increasingly blurred. Concepts from retrieval, generation, reinforcement
    learning, and even symbolic reasoning are often combined in complex pipelines. While this
    opens up new possibilities, it also introduces additional challenges for naive approaches
    that rely on simple assumptions about data and similarity.

    This mixture of relevant and irrelevant information is precisely what makes naive RAG
    systems struggle, and why more sophisticated approaches are necessary for robust performance.
    """
]

In [7]:
documents = [Document(page_content=text, metadata={"source": f"doc_{i}"}) for i, text in enumerate(raw_docs)]
print(f"Loaded {len(documents)} documents")

Loaded 6 documents


# Chunk the documents

> LLMs and embedding models work better on smaller pieces of text.

If documents are too large:

* retrieval becomes noisy
* embeddings become less precise
* prompt becomes too long

We split documents into chunks.

---
* each chunk has about 500 characters
* next chunk overlaps 100 characters
* overlap helps preserve context

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")
print("\nExample chunk:\n")
print(chunks[0].page_content)

Total chunks created: 27

Example chunk:

Retrieval-Augmented Generation (RAG) is a framework that combines information retrieval
    with large language model generation. It improves factual accuracy by grounding responses
    in external documents.


In [9]:
chunks[0]

Document(metadata={'source': 'doc_0'}, page_content='Retrieval-Augmented Generation (RAG) is a framework that combines information retrieval\n    with large language model generation. It improves factual accuracy by grounding responses\n    in external documents.')

In [10]:
chunks[4].metadata

{'source': 'doc_3'}

In [11]:
chunks[1].page_content

'Naive RAG typically includes document chunking, embedding generation, vector storage,\n    top-k retrieval, and answer generation using retrieved context.'

In [12]:
chunks[2].page_content

'A limitation of naive RAG is that it may retrieve irrelevant chunks if the embeddings\n    do not capture the true intent of the query.'

In [13]:
chunks[3].page_content

'In recent years, discussions around artificial intelligence have expanded beyond core\n    technical architectures into broader interdisciplinary domains. While Retrieval-Augmented\n    Generation (RAG) is often presented as a solution to hallucination problems in language\n    models, its practical implementation reveals a number of subtle challenges that are not\n    always obvious in simplified tutorials.'

In [14]:
chunks[4].page_content

'One such challenge arises when documents contain overlapping semantic signals. For example,\n    a paragraph discussing “retrieval efficiency” might also include unrelated mentions of\n    database indexing strategies, distributed systems, or even anecdotal examples from\n    unrelated industries like logistics or healthcare. When embeddings are generated, these\n    mixed signals can cause the vector representation to drift away from the primary intent'

In [15]:
chunks[5].page_content

'mixed signals can cause the vector representation to drift away from the primary intent\n    of the text. As a result, naive similarity search may rank such chunks higher than more\n    directly relevant content.'

# Create embeddings

In [16]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [17]:
embedding_model

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

# Build FAISS vector store

FAISS is a fast similarity search library.

We store chunk embeddings inside FAISS so we can retrieve similar chunks later.

In [18]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)
print("FAISS vector store created ")

FAISS vector store created 


# Create retriever

In [19]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Test retrieval only

In [20]:
query = "What is naive RAG?"

In [21]:
retrieved_docs = retriever.invoke(query)
retrieved_docs

[Document(id='aeded0ff-e936-4053-ab26-3135cc932493', metadata={'source': 'doc_1'}, page_content='Naive RAG typically includes document chunking, embedding generation, vector storage,\n    top-k retrieval, and answer generation using retrieved context.'),
 Document(id='3ea8f9fb-3be1-4d2b-a3d9-04c9fe474941', metadata={'source': 'doc_5'}, page_content='This mixture of relevant and irrelevant information is precisely what makes naive RAG\n    systems struggle, and why more sophisticated approaches are necessary for robust performance.'),
 Document(id='8189382e-32c1-43c3-b928-f778cd2f575c', metadata={'source': 'doc_2'}, page_content='A limitation of naive RAG is that it may retrieve irrelevant chunks if the embeddings\n    do not capture the true intent of the query.')]

In [22]:
print(f"Retrieved {len(retrieved_docs)} chunks\n")

Retrieved 3 chunks



In [23]:
for i, doc in enumerate(retrieved_docs, 1):
    print(f"--- Retrieved Chunk {i} ---")
    print(doc.page_content)
    print()

--- Retrieved Chunk 1 ---
Naive RAG typically includes document chunking, embedding generation, vector storage,
    top-k retrieval, and answer generation using retrieved context.

--- Retrieved Chunk 2 ---
This mixture of relevant and irrelevant information is precisely what makes naive RAG
    systems struggle, and why more sophisticated approaches are necessary for robust performance.

--- Retrieved Chunk 3 ---
A limitation of naive RAG is that it may retrieve irrelevant chunks if the embeddings
    do not capture the true intent of the query.



# Create LLM

In [24]:
# Initialize model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

In [25]:
llm

ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', temperature=0.2, client=<google.genai.client.Client object at 0x7dae89fbfef0>, default_metadata=(), model_kwargs={})

# Build prompt manually

In [26]:
def format_context(docs: List [Document]) -> str:
    return "\n\n".join([doc.page_content for doc in docs])

In [27]:
def build_rag_prompt(query: str, retrieved_docs: List[Document]) -> str:
    context = format_context(retrieved_docs)

    prompt = f"""
You are a helpful AI assistant.
Answer the user's question only using the context below.
If the answer is not found in the context, say: "I could not find the answer in the provided context."

Context:
{context}

Question:
{query}

Answer:
    """
    return prompt

# Generate answer

In [28]:
quary = "How does naive Rag work?"

In [29]:
retrieved_doc = retriever.invoke(query)

In [30]:
for i, doc in enumerate(retrieved_doc, 1):
    print(f"--- Retrieved Chunk {i} ---")
    print(doc.page_content)
    print()

--- Retrieved Chunk 1 ---
Naive RAG typically includes document chunking, embedding generation, vector storage,
    top-k retrieval, and answer generation using retrieved context.

--- Retrieved Chunk 2 ---
This mixture of relevant and irrelevant information is precisely what makes naive RAG
    systems struggle, and why more sophisticated approaches are necessary for robust performance.

--- Retrieved Chunk 3 ---
A limitation of naive RAG is that it may retrieve irrelevant chunks if the embeddings
    do not capture the true intent of the query.



In [31]:
prompt = build_rag_prompt(quary,retrieved_doc)

In [32]:
response = llm.invoke(prompt)

**AIMessage** is a class from LangChain. a response generated by the LLM

In [33]:
response

AIMessage(content='Naive RAG typically includes document chunking, embedding generation, vector storage, top-k retrieval, and answer generation using retrieved context.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d7331-2599-7b42-b444-964ac87de0a8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 155, 'output_tokens': 105, 'total_tokens': 260, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 78}})

In [34]:
response.content                                   # Main content

'Naive RAG typically includes document chunking, embedding generation, vector storage, top-k retrieval, and answer generation using retrieved context.'

In [35]:
response.response_metadata                         # Metadata (model info)

{'finish_reason': 'STOP',
 'model_name': 'gemini-2.5-flash',
 'safety_ratings': [],
 'model_provider': 'google_genai'}

In [36]:
print(response.id)
print(response.tool_calls)
print(response.invalid_tool_calls)


lc_run--019d7331-2599-7b42-b444-964ac87de0a8-0
[]
[]


In [37]:
response.usage_metadata                        # Metadata (model info)

{'input_tokens': 155,
 'output_tokens': 105,
 'total_tokens': 260,
 'input_token_details': {'cache_read': 0},
 'output_token_details': {'reasoning': 78}}

---

In this experiment, we explored the behavior of a Naive Retrieval-Augmented Generation (RAG) system under challenging conditions. By introducing long, noisy, and partially off-topic documents, we observed how the retrieval process can become unreliable when embeddings fail to accurately capture the true intent of a query.

The results highlight that while Naive RAG provides a simple and effective baseline, it is highly sensitive to data quality, chunking strategy, and semantic ambiguity. Irrelevant or mixed-context information can significantly degrade retrieval performance, leading to less accurate or incomplete responses.

### Problems of this naive RAG :

Common weaknesses

1. **Bad retrieval**: If embedding similarity is poor, wrong chunks are returned.

2. **Weak chunking**: Important context can be split badly.

3. **No reranker**: Top-k may include noisy chunks.

4. **No query rewriting**: Complex questions may retrieve poorly.

5. **No source validation**: Model may still hallucinate.

----

# RAG Evaluation Utilities

This section adds a **reusable evaluation layer** that you can keep the same from:

- Naive / Vanilla RAG
- Advanced RAG
- Hybrid RAG
- Graph RAG
- Agentic RAG

The idea is simple:

1. Define a **common test set format**
2. Define a **common prediction format**
3. Compute the **same metrics** for every RAG pipeline

---

## Metric groups used in this notebook

### 1) Retrieval metrics
These measure whether the retriever found the right document / chunk.

- Hit Rate@k
- Precision@k
- Recall@k
- MRR
- Average Precision@k
- nDCG@k

### 2) Answer metrics
These measure whether the generated answer matches the reference answer.

- Exact Match (EM)
- Token F1

### 3) System / production metrics
These measure runtime behavior.

- Retrieval latency
- Generation latency
- Total latency
- Empty retrieval rate
- Failure rate

### 4) Judge / manual metrics
These are important in real RAG evaluation, but they usually need:
- human annotation, or
- an LLM judge, or
- a framework like Ragas / LangSmith / DeepEval

Examples:
- Faithfulness / Groundedness
- Answer Relevance
- Context Relevance
- Context Precision
- Context Recall

In this notebook, we will implement the **classical reusable metrics directly in Python**, and also add a **manual evaluation template** so you can later extend it to all RAG variants.


In [38]:
import math
import re
import string
import time
from collections import Counter
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd


def normalize_text(text: str) -> str:
    """
    Normalize text for fair comparison.
    Lowercase, remove punctuation, and collapse extra spaces.
    """
    if text is None:
        return ""
    
    text = text.lower().strip()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text)
    return text


# Checks if the predicted answer exactly matches the reference answer after normalization.
def exact_match_score(prediction: str, reference: str) -> float:
    """
    Exact Match after normalization.
    Returns 1.0 if equal, else 0.0.
    """
    return float(normalize_text(prediction) == normalize_text(reference))


# Measures word overlap between prediction and reference.
def token_f1_score(prediction: str, reference: str) -> float:
    """
    Token-level F1 between predicted answer and reference answer.
    """
    pred_tokens = normalize_text(prediction).split()
    ref_tokens = normalize_text(reference).split()

    if len(pred_tokens) == 0 and len(ref_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    pred_counter = Counter(pred_tokens)
    ref_counter = Counter(ref_tokens)

    common = pred_counter & ref_counter
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(ref_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)



#  --- Hit Rate@k
# Checks whether at least one correct item appears in top-k retrieved results.
def hit_rate_at_k(retrieved_ids: List[str], gold_ids: List[str], k: int) -> float:
    """
    Returns 1.0 if at least one gold item appears in top-k retrieved IDs.
    """
    retrieved_top_k = retrieved_ids[:k]
    return float(any(item in gold_ids for item in retrieved_top_k))



# ---Precision@k
# Among the top-k retrieved items, how many are actually correct?
def precision_at_k(retrieved_ids: List[str], gold_ids: List[str], k: int) -> float:
    """
    Precision@k = relevant retrieved in top-k / k
    """
    retrieved_top_k = retrieved_ids[:k]
    if k == 0:
        return 0.0

    hits = sum(1 for item in retrieved_top_k if item in gold_ids)
    return hits / k


# ---Recall@k
# Among all truly relevant items, how many did retrieval find in top-k?
def recall_at_k(retrieved_ids: List[str], gold_ids: List[str], k: int) -> float:
    """
    Recall@k = relevant retrieved in top-k / total relevant items
    """
    if len(gold_ids) == 0:
        return 0.0

    retrieved_top_k = retrieved_ids[:k]
    hits = sum(1 for item in retrieved_top_k if item in gold_ids)
    return hits / len(gold_ids)



# ---MRR (Mean Reciprocal Rank component)
# Looks at the rank of the first correct result.
def reciprocal_rank(retrieved_ids: List[str], gold_ids: List[str]) -> float:
    """
    Reciprocal Rank for a single query.
    """
    for rank, item in enumerate(retrieved_ids, start=1):
        if item in gold_ids:
            return 1.0 / rank
    return 0.0



# ---DCG and nDCG@k
# Measures ranking quality, giving more reward when relevant items appear near the top.
def dcg_at_k(retrieved_ids: List[str], gold_ids: List[str], k: int) -> float:
    """
    Discounted Cumulative Gain at k.
    Binary relevance: 1 if item is relevant, else 0.
    """
    retrieved_top_k = retrieved_ids[:k]
    dcg = 0.0

    for i, item in enumerate(retrieved_top_k, start=1):
        rel = 1 if item in gold_ids else 0
        if rel > 0:
            dcg += rel / math.log2(i + 1)

    return dcg


def ndcg_at_k(retrieved_ids: List[str], gold_ids: List[str], k: int) -> float:
    """
    Normalized DCG at k.
    """
    actual_dcg = dcg_at_k(retrieved_ids, gold_ids, k)

    ideal_retrieved = gold_ids[:k]
    ideal_dcg = dcg_at_k(ideal_retrieved, gold_ids, k)

    if ideal_dcg == 0:
        return 0.0

    return actual_dcg / ideal_dcg



# ---Average Precision@k
# It looks at precision across ranks where relevant items appear.
def average_precision_at_k(retrieved_ids: List[str], gold_ids: List[str], k: int) -> float:
    """
    Average Precision at k for binary relevance.
    """
    retrieved_top_k = retrieved_ids[:k]

    if len(gold_ids) == 0:
        return 0.0

    precisions = []
    hit_count = 0

    for idx, item in enumerate(retrieved_top_k, start=1):
        if item in gold_ids:
            hit_count += 1
            precisions.append(hit_count / idx)

    if not precisions:
        return 0.0

    return sum(precisions) / min(len(gold_ids), k)


# ---Manual metric template
# Creates a place to record judge-style evaluations manually for now.
def manual_judge_template(
    faithfulness: Optional[float] = None,
    answer_relevance: Optional[float] = None,
    context_relevance: Optional[float] = None,
    context_precision_manual: Optional[float] = None,
    context_recall_manual: Optional[float] = None
) -> Dict[str, Optional[float]]:
    """
    Placeholder for manual or later LLM-judge metrics.
    """
    return {
        "faithfulness": faithfulness,
        "answer_relevance": answer_relevance,
        "context_relevance": context_relevance,
        "context_precision_manual": context_precision_manual,
        "context_recall_manual": context_recall_manual
    }



# ---Failure flags
#Creates simple production-style flags.
def build_failure_flags(
    retrieved_chunk_ids: List[str],
    gold_chunk_ids: List[str],
    predicted_answer: str,
    reference_answer: str,
    total_latency_sec: Optional[float] = None,
    latency_threshold_sec: float = 5.0
) -> Dict[str, int]:
    """
    Create simple failure indicators.
    """
    failed_retrieval = int(hit_rate_at_k(retrieved_chunk_ids, gold_chunk_ids, k=5) == 0)
    failed_answer = int(
        exact_match_score(predicted_answer, reference_answer) == 0 and
        token_f1_score(predicted_answer, reference_answer) < 0.3
    )
    high_latency = int((total_latency_sec or 0.0) > latency_threshold_sec)

    return {
        "failed_retrieval": failed_retrieval,
        "failed_answer": failed_answer,
        "high_latency": high_latency
    }



# ---Evaluate one prediction
#Evaluates one question-answer result.
def evaluate_single_prediction(
    sample: Dict[str, Any],
    prediction: Dict[str, Any],
    k_values: List[int] = [1, 3, 5]
) -> Dict[str, Any]:
    """
    Evaluate one RAG output against one gold sample.
    """
    reference_answer = sample.get("reference_answer", "")
    gold_document_ids = sample.get("gold_document_ids", [])
    gold_chunk_ids = sample.get("gold_chunk_ids", [])

    predicted_answer = prediction.get("predicted_answer", "")
    retrieved_document_ids = prediction.get("retrieved_document_ids", [])
    retrieved_chunk_ids = prediction.get("retrieved_chunk_ids", [])

    row = {
        "question_id": sample.get("id"),
        "question": sample.get("question"),
        "reference_answer": reference_answer,
        "predicted_answer": predicted_answer,

        # Answer metrics
        "exact_match": exact_match_score(predicted_answer, reference_answer),
        "token_f1": token_f1_score(predicted_answer, reference_answer),

        # Latency
        "retrieval_latency_sec": prediction.get("retrieval_latency_sec"),
        "generation_latency_sec": prediction.get("generation_latency_sec"),
        "total_latency_sec": prediction.get("total_latency_sec"),
    }

    # Retrieval metrics: document level and chunk level
    for k in k_values:
        row[f"doc_hit_rate@{k}"] = hit_rate_at_k(retrieved_document_ids, gold_document_ids, k)
        row[f"doc_precision@{k}"] = precision_at_k(retrieved_document_ids, gold_document_ids, k)
        row[f"doc_recall@{k}"] = recall_at_k(retrieved_document_ids, gold_document_ids, k)
        row[f"doc_ndcg@{k}"] = ndcg_at_k(retrieved_document_ids, gold_document_ids, k)
        row[f"doc_ap@{k}"] = average_precision_at_k(retrieved_document_ids, gold_document_ids, k)

        row[f"chunk_hit_rate@{k}"] = hit_rate_at_k(retrieved_chunk_ids, gold_chunk_ids, k)
        row[f"chunk_precision@{k}"] = precision_at_k(retrieved_chunk_ids, gold_chunk_ids, k)
        row[f"chunk_recall@{k}"] = recall_at_k(retrieved_chunk_ids, gold_chunk_ids, k)
        row[f"chunk_ndcg@{k}"] = ndcg_at_k(retrieved_chunk_ids, gold_chunk_ids, k)
        row[f"chunk_ap@{k}"] = average_precision_at_k(retrieved_chunk_ids, gold_chunk_ids, k)

    row["doc_mrr"] = reciprocal_rank(retrieved_document_ids, gold_document_ids)
    row["chunk_mrr"] = reciprocal_rank(retrieved_chunk_ids, gold_chunk_ids)

    # Optional manual judge scores
    manual_scores = prediction.get("manual_judge_scores", {})
    row["faithfulness"] = manual_scores.get("faithfulness")
    row["answer_relevance"] = manual_scores.get("answer_relevance")
    row["context_relevance"] = manual_scores.get("context_relevance")
    row["context_precision_manual"] = manual_scores.get("context_precision_manual")
    row["context_recall_manual"] = manual_scores.get("context_recall_manual")

    # Failure flags
    row.update(
        build_failure_flags(
            retrieved_chunk_ids=retrieved_chunk_ids,
            gold_chunk_ids=gold_chunk_ids,
            predicted_answer=predicted_answer,
            reference_answer=reference_answer,
            total_latency_sec=prediction.get("total_latency_sec")
        )
    )

    return row


# --Evaluate many predictions
#Runs evaluation over the full test set.
def evaluate_predictions(
    samples: List[Dict[str, Any]],
    predictions: List[Dict[str, Any]],
    k_values: List[int] = [1, 3, 5]
) -> pd.DataFrame:
    """
    Evaluate a full list of predictions against samples.
    """
    pred_map = {pred["question_id"]: pred for pred in predictions}
    rows = []

    for sample in samples:
        qid = sample["id"]
        if qid not in pred_map:
            continue

        row = evaluate_single_prediction(
            sample=sample,
            prediction=pred_map[qid],
            k_values=k_values
        )
        rows.append(row)

    return pd.DataFrame(rows)


# ----Summary table --- Creates a compact average summary over all numeric metrics.
def summarize_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Mean summary over numeric columns.
    """
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    summary = df[numeric_cols].mean().to_frame(name="mean").reset_index()
    summary.columns = ["metric", "mean"]
    return summary.sort_values("metric").reset_index(drop=True)


# --Failure breakdown=----------------------
def failure_breakdown(df: pd.DataFrame) -> pd.DataFrame:
    """
    Count major failure flags.
    """
    cols = ["failed_retrieval", "failed_answer", "high_latency"]
    existing_cols = [c for c in cols if c in df.columns]

    breakdown = []
    total = len(df)

    for col in existing_cols:
        count = int(df[col].sum())
        rate = count / total if total > 0 else 0.0
        breakdown.append({
            "failure_type": col,
            "count": count,
            "rate": rate
        })

    return pd.DataFrame(breakdown)



# ----------------------Utils ----------------------
def extract_retrieval_ids_from_docs(retrieved_docs: List[Any]) -> Dict[str, List[str]]:
    """
    Extract document and chunk IDs from retrieved docs.
    Assumes each doc.metadata contains:
    - document_id
    - chunk_id
    """
    document_ids = []
    chunk_ids = []
    chunk_texts = []

    for doc in retrieved_docs:
        document_ids.append(doc.metadata.get("document_id", "unknown_document"))
        chunk_ids.append(doc.metadata.get("chunk_id", "unknown_chunk"))
        chunk_texts.append(doc.page_content)

    return {
        "retrieved_document_ids": document_ids,
        "retrieved_chunk_ids": chunk_ids,
        "retrieved_chunks": chunk_texts
    }

def add_ids_to_chunks(chunks: List[Any]) -> List[Any]:
    """
    Add stable document_id and chunk_id to each chunk.
    Uses file_name + page if available.
    """
    for idx, chunk in enumerate(chunks):
        file_name = chunk.metadata.get("file_name", "unknown_file")
        page = chunk.metadata.get("page", "unknown_page")

        document_id = file_name
        chunk_id = f"{file_name}::page_{page}::chunk_{idx}"

        chunk.metadata["document_id"] = document_id
        chunk.metadata["chunk_id"] = chunk_id

    return chunks

## EXAMPLE TEST SET

In [51]:
def add_ids_to_chunks(chunks):
    """
    Add stable document_id and chunk_id to each chunk.
    Works for raw_docs-based datasets.
    """
    for idx, chunk in enumerate(chunks):
        source = chunk.metadata.get("source", "unknown_source")

        document_id = source
        chunk_id = f"{source}::chunk_{idx}"

        chunk.metadata["document_id"] = document_id
        chunk.metadata["chunk_id"] = chunk_id

    return chunks

## Step 1 — Convert raw_docs into LangChain Document objects

In [52]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=text,
        metadata={"source": f"doc_{i}"}
    )
    for i, text in enumerate(raw_docs)
]

print(f"Total documents: {len(documents)}")
print(documents[0].metadata)
print(documents[0].page_content[:300])

Total documents: 6
{'source': 'doc_0'}

    Retrieval-Augmented Generation (RAG) is a framework that combines information retrieval
    with large language model generation. It improves factual accuracy by grounding responses
    in external documents.
    


## Step 2 — Chunking

In [53]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")
print("\nExample chunk:\n")
print(chunks[0].page_content)
print("\nExample metadata:\n")
print(chunks[0].metadata)

Total chunks created: 27

Example chunk:

Retrieval-Augmented Generation (RAG) is a framework that combines information retrieval
    with large language model generation. It improves factual accuracy by grounding responses
    in external documents.

Example metadata:

{'source': 'doc_0'}


In [55]:
chunks = add_ids_to_chunks(chunks)

print(chunks[0].metadata)
print(chunks[1].metadata)

{'source': 'doc_0', 'document_id': 'doc_0', 'chunk_id': 'doc_0::chunk_0'}
{'source': 'doc_1', 'document_id': 'doc_1', 'chunk_id': 'doc_1::chunk_1'}


In [56]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [57]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("FAISS vector store created")

FAISS vector store created


In [58]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [59]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7dae89fbff20>, search_kwargs={'k': 4})

In [64]:
# Test retrieval normally
query = "How does naive RAG work?"
retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"--- Retrieved Chunk {i} ---")
    print(doc.metadata)
    print(doc.page_content[:300])
    print()

--- Retrieved Chunk 1 ---
{'source': 'doc_5', 'document_id': 'doc_5', 'chunk_id': 'doc_5::chunk_26'}
This mixture of relevant and irrelevant information is precisely what makes naive RAG
    systems struggle, and why more sophisticated approaches are necessary for robust performance.

--- Retrieved Chunk 2 ---
{'source': 'doc_1', 'document_id': 'doc_1', 'chunk_id': 'doc_1::chunk_1'}
Naive RAG typically includes document chunking, embedding generation, vector storage,
    top-k retrieval, and answer generation using retrieved context.

--- Retrieved Chunk 3 ---
{'source': 'doc_4', 'document_id': 'doc_4', 'chunk_id': 'doc_4::chunk_16'}
Additionally, naive RAG systems typically rely on static chunk sizes. This can lead to
    situations where important context is split across chunks, reducing its usefulness, while
    less relevant information is grouped together and retrieved as a single unit. For example,
    a detailed explanati

--- Retrieved Chunk 4 ---
{'source': 'doc_2', 'document_

In [67]:
# Generate answer
prompt = build_rag_prompt(query, retrieved_docs)

generation_start = time.time()
response = llm.invoke(prompt)
generation_latency = time.time() - generation_start

predicted_answer = response.content
total_latency = retrieval_latency + generation_latency

print("Predicted Answer:\n")
print(predicted_answer)

Predicted Answer:

Naive RAG typically includes document chunking, embedding generation, vector storage, top-k retrieval, and answer generation using retrieved context.


In [69]:
sample = {
    "id": "q1",
    "question": query,
    "reference_answer": "Naive RAG typically includes document chunking, embedding generation, vector storage, top-k retrieval, and answer generation using retrieved context.",
    "gold_document_ids": ["doc_1"],
    "gold_chunk_ids": ["doc_1::chunk_1"]  
}

In [70]:
retrieval_info = extract_retrieval_ids_from_docs(retrieved_docs)
print(retrieval_info)

{'retrieved_document_ids': ['doc_5', 'doc_1', 'doc_4', 'doc_2'], 'retrieved_chunk_ids': ['doc_5::chunk_26', 'doc_1::chunk_1', 'doc_4::chunk_16', 'doc_2::chunk_2'], 'retrieved_chunks': ['This mixture of relevant and irrelevant information is precisely what makes naive RAG\n    systems struggle, and why more sophisticated approaches are necessary for robust performance.', 'Naive RAG typically includes document chunking, embedding generation, vector storage,\n    top-k retrieval, and answer generation using retrieved context.', 'Additionally, naive RAG systems typically rely on static chunk sizes. This can lead to\n    situations where important context is split across chunks, reducing its usefulness, while\n    less relevant information is grouped together and retrieved as a single unit. For example,\n    a detailed explanation of RAG limitations might be split into two chunks, each of which\n    appears incomplete on its own. Meanwhile, a chunk containing a mix of unrelated topics', 'A 

In [71]:
prediction = {
    "question_id": "q1",
    "predicted_answer": predicted_answer,
    "retrieved_document_ids": retrieval_info["retrieved_document_ids"],
    "retrieved_chunk_ids": retrieval_info["retrieved_chunk_ids"],
    "retrieved_chunks": retrieval_info["retrieved_chunks"],
    "retrieval_latency_sec": retrieval_latency,
    "generation_latency_sec": generation_latency,
    "total_latency_sec": total_latency,
    "manual_judge_scores": manual_judge_template()
}

In [72]:
eval_row = evaluate_single_prediction(sample, prediction, k_values=[1, 3, 5])

for key, value in eval_row.items():
    print(f"{key}: {value}")

question_id: q1
question: How does naive RAG work?
reference_answer: Naive RAG typically includes document chunking, embedding generation, vector storage, top-k retrieval, and answer generation using retrieved context.
predicted_answer: Naive RAG typically includes document chunking, embedding generation, vector storage, top-k retrieval, and answer generation using retrieved context.
exact_match: 1.0
token_f1: 1.0
retrieval_latency_sec: 0.020601987838745117
generation_latency_sec: 0.8672909736633301
total_latency_sec: 0.8878929615020752
doc_hit_rate@1: 0.0
doc_precision@1: 0.0
doc_recall@1: 0.0
doc_ndcg@1: 0.0
doc_ap@1: 0.0
chunk_hit_rate@1: 0.0
chunk_precision@1: 0.0
chunk_recall@1: 0.0
chunk_ndcg@1: 0.0
chunk_ap@1: 0.0
doc_hit_rate@3: 1.0
doc_precision@3: 0.3333333333333333
doc_recall@3: 1.0
doc_ndcg@3: 0.6309297535714575
doc_ap@3: 0.5
chunk_hit_rate@3: 1.0
chunk_precision@3: 0.3333333333333333
chunk_recall@3: 1.0
chunk_ndcg@3: 0.6309297535714575
chunk_ap@3: 0.5
doc_hit_rate@5: 1.0
d

---

# 👨‍💻 Author
# **Md Saib Hossain**
**AI Engineer • AI / ML / LLM & AI Safety Researcher**  
**Agentic AI Developer • Researcher in Autonomous & Multi-Agent Systems • Advanced Agentic AI Architect**

Designing safe, scalable, and human-centered intelligent systems for real-world healthcare and autonomous AI applications.

<p align="left">
  <a href="mailto:saibhossain5@gmail.com">
    <img src="https://img.shields.io/badge/Email-saibhossain5%40gmail.com-red?style=flat&logo=gmail">
  </a>
  <a href="https://saibhossain.github.io/">
    <img src="https://img.shields.io/badge/Portfolio-Visit-blue?style=flat&logo=google-chrome">
  </a>
  <a href="https://github.com/Saibhossain">
    <img src="https://img.shields.io/badge/GitHub-Profile-black?style=flat&logo=github">
  </a>
  <a href="https://linkedin.com/in/saib-hossain-182834229">
    <img src="https://img.shields.io/badge/LinkedIn-Connect-0A66C2?style=flat&logo=linkedin">
  </a>
</p>
